In [ ]:
pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.2 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

model = YOLO("/content/best.pt")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
results = model.predict(
    source="test.mp4",
    save=True
)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/505) /content/test.mp4: 384x640 2 COWs, 308.3ms
video 1/1 (frame 2/505) /content/test.mp4: 384x640 2 COWs, 120.0ms
video 1/1 (frame 3/505) /content/test.mp4: 384x640 2 COWs, 115.5ms
video 1/1 (frame 4/505) /content/test.mp4: 384x640 2 COWs, 121.7ms
video 1/1 (frame 5/505) /content/test.mp4: 384x640 2 COWs, 126.7ms
video 1/1 (frame 6/505) /content/test.mp4: 384x640 2 COWs, 117.4ms
video 1/1 (frame 7/505) /content/test.mp4: 384x640 2 COW

KeyboardInterrupt: 

# Evaluate

In [ ]:
from ultralytics import YOLO

model = YOLO("best.pt")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
metrics = model.val(data="/content/CV-PROJECT-4-C.v5-good-aug.yolov8/data.yaml")

Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 73 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 599.0±388.3 MB/s, size: 56.1 KB)
val: Scanning /content/CV-PROJECT-4-C.v5-good-aug.yolov8/valid/labels... 1195 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1195/1195 777.2it/s 1.5s
val: New cache created: /content/CV-PROJECT-4-C.v5-good-aug.yolov8/valid/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 75/75 4.5s/it 5:41
                   all       1195       4012      0.761      0.732      0.757      0.683
                   COW        474        663      0.974      0.977      0.987      0.964
                 horse        103        110      0.995      0.936      0.954      0.892
                   pig        342       2141      0.985      0.981      0.993      0.895
                 sheep       

In [ ]:
!zip -r /content/cow_detect_val.zip /content/runs/detect/val

  adding: content/runs/detect/val/ (stored 0%)
  adding: content/runs/detect/val/BoxR_curve.png (deflated 10%)
  adding: content/runs/detect/val/val_batch0_labels.jpg (deflated 4%)
  adding: content/runs/detect/val/confusion_matrix_normalized.png (deflated 24%)
  adding: content/runs/detect/val/val_batch1_pred.jpg (deflated 6%)
  adding: content/runs/detect/val/val_batch2_labels.jpg (deflated 6%)
  adding: content/runs/detect/val/val_batch1_labels.jpg (deflated 6%)
  adding: content/runs/detect/val/confusion_matrix.png (deflated 25%)
  adding: content/runs/detect/val/BoxP_curve.png (deflated 11%)
  adding: content/runs/detect/val/val_batch0_pred.jpg (deflated 5%)
  adding: content/runs/detect/val/val_batch2_pred.jpg (deflated 7%)
  adding: content/runs/detect/val/BoxPR_curve.png (deflated 13%)
  adding: content/runs/detect/val/BoxF1_curve.png (deflated 10%)


In [ ]:
pip install deep-sort-realtime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 45.0 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO
import cv2
from deep_sort_realtime.deepsort_tracker import DeepSort

In [ ]:
from ultralytics import YOLO
import cv2
from deep_sort_realtime.deepsort_tracker import DeepSort
from google.colab.patches import cv2_imshow

model = YOLO("best.pt")
tracker = DeepSort()

frame_skip = 5   # process every 5th frame
frame_count = 0

cap = cv2.VideoCapture("test.mp4")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1
    if frame_count % frame_skip != 0:
        continue

    results = model(frame)[0]

    detections = []
    for box in results.boxes:
        x1,y1,x2,y2 = box.xyxy[0]
        conf = box.conf[0]
        detections.append(([x1,y1,x2-x1,y2-y1], conf, 'cow'))

    tracks = tracker.update_tracks(detections, frame=frame)

    for track in tracks:
        if not track.is_confirmed():
            continue
        track_id = track.track_id
        l,t,r,b = map(int, track.to_ltrb())

        cow_crop = frame[t:b, l:r]   # crop cow
        # Check if the cropped region is valid (has positive dimensions)
        if cow_crop.shape[0] > 0 and cow_crop.shape[1] > 0:
          cv2_imshow(cow_crop)
        cv2.rectangle(frame,(l,t),(r,b),(0,255,0),2)
        cv2.putText(frame,f"ID {track_id}",(l,t-5),0,0.6,(0,255,0),2)

    # cv2_imshow(frame)